# Phase 2: Intermediate Python — Writing Modular, Pythonic AI Code

Welcome to Phase 2! Now that you know variables, loops, lists, and functions from Phase 1, it is time to learn how professional AI developers structure real-world projects. Every topic is explained clearly from the ground up before showing the code.

### What You Will Learn in Phase 2:
1. **Comprehensions & Functional Tools** (List/Dict/Set Comprehensions, `map`, `filter`, `functools.reduce`, `lru_cache`, `partial`)[cite: 1, 2, 3, 4, 5]
2. **Iterators, Generators & `itertools`** (`iter`, `next`, `yield` for streaming huge datasets without crashing RAM)[cite: 2, 4, 5]
3. **Object-Oriented Programming (OOP), Dunder Methods & `dataclasses`** (Classes, Inheritance, `@property`, `@classmethod`, `@staticmethod`, `__call__`, `__getitem__`, and introspection like `isinstance`, `getattr`, `hasattr`)[cite: 1, 3, 4, 5]
4. **Exception Handling, Debugging & Context Managers** (`try/except/else/finally`, custom errors, and `with` statements)[cite: 1]
5. **File I/O, Directory Paths & Data Persistence** (`open`, `pathlib`, `os`, `json`, `csv`, `yaml`, `tomllib`, `tempfile`, `zipfile`, `sqlite3`, `shelve`)[cite: 1, 2, 4]
6. **Standard Library Power Tools** (`collections`, `re`, `hashlib`, `random`, `math`, `statistics`, `datetime`, `zoneinfo`, `argparse`, `logging`, `urllib.request`)[cite: 1, 2]

---
## Lesson 1: Comprehensions & Functional Tools (`map`, `filter`, `functools`)[cite: 1, 2, 3, 4, 5]

### What is a Comprehension?[cite: 1]
In Phase 1, if you wanted to create a new list from an old list, you wrote a 4-line `for` loop using `.append()`. A **List Comprehension** lets you write that exact same loop in **one clean, fast line of code**:
* **Formula:** `[expression for item in iterable if condition]`
* You can also do the exact same thing for Dictionaries (`{k: v for ...}`) and Sets (`{item for ...}`).

### Functional Built-ins (`map`, `filter`, `functools`)[cite: 2, 3, 4, 5]:
* **`map(function, list)`**[cite: 4, 5]: Applies a function to every single item in a list.
* **`filter(function, list)`**[cite: 3, 5]: Keeps only the items where the function returns `True`.
* **`functools.lru_cache`**[cite: 2]: A memory-saver! If you call a slow function with the exact same input twice, `@lru_cache` remembers the previous answer and returns it instantly without recalculating.

In [ ]:
from functools import reduce, lru_cache, partial

# --- 1. The Old Way (4 lines) vs. List Comprehension (1 line) ---
raw_words = ["  Hello ", "WORLD", "  ai  ", "PyTorch"]

# Old way:
cleaned_old = []
for w in raw_words:
    cleaned_old.append(w.strip().lower())

# Pythonic Way (List Comprehension):
cleaned_new = [w.strip().lower() for w in raw_words]
print("List Comprehension:", cleaned_new)

# Comprehension with an 'if' filter (keep only words longer than 2 letters):
long_words = [w for w in cleaned_new if len(w) > 2]
print("Filtered List Comprehension:", long_words)

# Dictionary Comprehension (Mapping word -> character count):
word_lengths = {w: len(w) for w in long_words}
print("Dictionary Comprehension:", word_lengths)

# --- 2. map(), filter(), and functools.reduce() ---
probabilities = [0.2, 0.85, 0.91, 0.4, 0.77]

# filter() keeps probabilities >= 0.75
confident_probs = list(filter(lambda p: p >= 0.75, probabilities))
# map() converts decimals to percentages
percentages = list(map(lambda p: round(p * 100, 1), confident_probs))
# reduce() multiplies all probabilities together
combined_prob = reduce(lambda a, b: a * b, confident_probs)

print("Confident Percentages:", percentages, "| Combined Prob:", round(combined_prob, 4))

# --- 3. functools.lru_cache (Caching expensive AI calls) ---
@lru_cache(maxsize=100)
def slow_token_lookup(word: str) -> int:
    print(f"  -> Computing token ID for '{word}' from scratch...")
    return sum(ord(c) for c in word)

print("First call:", slow_token_lookup("transformer"))
print("Second call (Instant from cache!):", slow_token_lookup("transformer"))

---
## Lesson 2: Iterators, Generators (`yield`) & `itertools`[cite: 2, 4, 5]

### Why Do AI Developers Need Generators?
Imagine you are training an AI model on **500 Gigabytes of images or text**, but your laptop only has **16 GB of RAM**. If you try to load all 500 GB into a standard Python `list`, your computer will immediately freeze and crash!

### How `yield` Solves This:
A normal function uses `return` to build the entire result in memory at once and finish. A **Generator function** uses the keyword **`yield`**. It produces **just one batch of data at a time**, pauses its state, waits for your training loop to ask for the `next()` batch, and then continues! It uses almost zero RAM.

In [ ]:
import itertools

# 1. Writing a Generator Function with 'yield'
def data_batch_streamer(total_rows: int, batch_size: int):
    """Streams small batches of data one at a time without filling up RAM."""
    for start_idx in range(0, total_rows, batch_size):
        # Create only the current small slice of data
        batch = [f"row_{i}" for i in range(start_idx, min(start_idx + batch_size, total_rows))]
        yield batch  # Pauses here and hands the batch to the caller!

# Calling the generator function does NOT run the code yet; it creates a generator object
streamer = data_batch_streamer(total_rows=7, batch_size=3)

# We can pull the first batch manually using next()
print("Manual 1st pull via next():", next(streamer))

# Or loop over the remaining batches automatically:
for batch in streamer:
    print("Looped batch:", batch)

# 2. itertools: Built-in tools for advanced iteration
# itertools.product creates every possible combination (Used for Hyperparameter Grid Search!)
models = ["BERT", "RoBERTa"]
learning_rates = [0.01, 0.001]
experiments = list(itertools.product(models, learning_rates))
print("All Experiment Combinations:", experiments)

---
## Lesson 3: Object-Oriented Programming (OOP), Dunder Methods & `dataclasses`[cite: 1, 3, 4, 5]

### What is OOP (Classes vs. Objects)?[cite: 1]
* **Class:** A blueprint or cookie-cutter (e.g., `class NeuralNetwork`). It defines what data (**attributes**) and actions (**methods**) something has.
* **Object (Instance):** The actual cookie made from that blueprint.
* **`self`:** Inside a class, `self` refers to the specific object being created or used.

### Why Dunder (Double Underscore) Methods Matter in PyTorch:
Special methods surrounded by double underscores (`__init__`, `__len__`, `__getitem__`, `__call__`) let your custom objects act like built-in Python types:
* **`__init__(self)`**: Runs automatically when you create a new object.
* **`__call__(self)`**: Lets you call your object like a function (`model(input_data)`). **Every PyTorch neural network uses this!**
* **`__len__(self)` & `__getitem__(self, idx)`**: Let you use `len(dataset)` and `dataset[0]`. **Every PyTorch Dataset uses these!**

### What is a `@dataclass`?[cite: 1]
If you just need a class to hold configuration settings (like `epochs`, `batch_size`, `learning_rate`), writing `__init__` manually is tedious. Adding `@dataclass` above your class writes `__init__` and `__repr__` for you automatically!

In [ ]:
from dataclasses import dataclass

# --- 1. Dataclasses: Clean containers for AI Settings ---
@dataclass
class ModelConfig:
    architecture: str
    hidden_size: int = 768
    dropout: float = 0.1

cfg = ModelConfig(architecture="Transformer")
print("Dataclass Object:", cfg)

# --- 2. Full OOP Class with Inheritance, Decorators & Dunder Methods ---
class BaseComponent:
    def __init__(self, name: str):
        self.name = name

class SimpleLinearLayer(BaseComponent):  # Inherits from BaseComponent
    def __init__(self, name: str, weight: float, bias: float):
        super().__init__(name)           # super() calls the parent class's __init__
        self._weight = weight
        self.bias = bias

    @property                            # Lets you access .weight like a variable, not a function
    def weight(self) -> float:
        return self._weight

    @classmethod                         # Creates an instance from a dictionary
    def from_dict(cls, data: dict):
        return cls(data["name"], data["weight"], data["bias"])

    @staticmethod                        # Utility function that doesn't need 'self'
    def activate(val: float) -> float:
        return max(0.0, val)             # ReLU activation: turns negatives into 0

    def __call__(self, x: float) -> float:
        # Allows us to run layer(5.0) directly like a function!
        raw_output = (x * self._weight) + self.bias
        return self.activate(raw_output)

    def __repr__(self) -> str:
        return f"SimpleLinearLayer(name='{self.name}', weight={self._weight}, bias={self.bias})"

# Creating and calling our layer
layer = SimpleLinearLayer.from_dict({"name": "dense_1", "weight": 2.5, "bias": -1.0})
print("Layer Info:", repr(layer))
print("Forward Pass Output for x=4.0:", layer(4.0))  # Triggers __call__!

# --- 3. Object Introspection Built-ins (isinstance, hasattr, getattr, setattr, callable) ---
print("Is it a BaseComponent?:", isinstance(layer, BaseComponent))
print("Can we call it like a function?:", callable(layer))
if hasattr(layer, "bias"):
    print("Current bias via getattr():", getattr(layer, "bias"))
    setattr(layer, "bias", 0.0)          # Dynamically updates layer.bias to 0.0
    print("Updated bias:", layer.bias)

---
## Lesson 4: Exception Handling (`try/except`) & Context Managers (`with`)[cite: 1]

### Why Exception Handling Matters in AI:
Imagine your code is calling an LLM API 1,000 times, or reading 10,000 images from disk. What if image #4,502 is corrupted, or the internet drops for 1 second? Without **Exception Handling**, your entire program crashes and you lose hours of progress!
* **`try`:** Put risky code here (like network calls or file reading).
* **`except`:** Runs if an error happens inside `try`, letting your program recover gracefully.
* **`else`:** Runs ONLY if no error happened.
* **`finally`:** Runs no matter what (great for closing connections or freeing GPU memory).

### What is a Context Manager (`with`)?[cite: 1]
Whenever you open a file or temporarily disable PyTorch gradients (`with torch.no_grad():`), the `with` statement guarantees that cleanup happens automatically as soon as the indented block finishes—even if an error occurs inside!

In [ ]:
from contextlib import contextmanager

# 1. Handling Errors Safely with try / except / else / finally
raw_api_outputs = ['{"score": 0.95}', 'CORRUPTED_JSON_TEXT', '{"score": 0.82}']

import json
valid_scores = []

for item in raw_api_outputs:
    try:
        parsed = json.loads(item)        # Risky operation: might fail on corrupted text
    except json.JSONDecodeError as err:
        print(f"  [Warning] Skipping corrupted item: '{item}'")
    else:
        valid_scores.append(parsed["score"])

print("Successfully recovered scores:", valid_scores)

# 2. Creating a Custom Context Manager (Simulating PyTorch's 'with no_grad():')
@contextmanager
def evaluation_mode(model_name: str):
    print(f"\n[Entering Eval Mode for {model_name} - Gradients Disabled]")
    try:
        yield  # Code inside the 'with' block runs here
    finally:
        print(f"[Exiting Eval Mode for {model_name} - Gradients Re-enabled]")

with evaluation_mode("ResNet-50"):
    print("  -> Running fast validation predictions...")

---
## Lesson 5: Files, Paths, Configs & Local Databases[cite: 1, 2, 4]

AI developers work with many file types every day:
* **`pathlib.Path`**[cite: 1, 2]: The modern, object-oriented way to handle file and folder paths across Windows, Linux, and Mac.
* **`json` & `yaml` & `tomllib`**[cite: 1, 2]: Used for LLM API payloads, prompt templates, and model configuration files.
* **`csv`**[cite: 2]: Tabular spreadsheet data (rows and columns).
* **`tempfile` & `zipfile`**[cite: 2]: Creating temporary workspaces and extracting zipped datasets.
* **`sqlite3` & `shelve`**[cite: 2]: Built-in databases that require zero server setup—perfect for saving chatbot conversation histories or caching LLM responses locally!

In [ ]:
import csv
import json
import sqlite3
import tempfile
import tomllib
import zipfile
from pathlib import Path

# We use TemporaryDirectory so this notebook cleans up after itself automatically!
with tempfile.TemporaryDirectory() as temp_folder:
    folder_path = Path(temp_folder)

    # 1. Writing & Reading a JSON file
    config_file = folder_path / "config.json"
    config_data = {"model": "Llama-3", "temperature": 0.7}
    config_file.write_text(json.dumps(config_data, indent=2), encoding="utf-8")
    print("Read JSON back:", json.loads(config_file.read_text(encoding="utf-8")))

    # 2. Writing & Reading a CSV dataset
    csv_file = folder_path / "prompts.csv"
    with open(csv_file, mode="w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["id", "text", "Category"])
        writer.writerow([1, "How to learn Python?", "Education"])

    # 3. Compressing files into a .zip archive
    zip_file = folder_path / "dataset_bundle.zip"
    with zipfile.ZipFile(zip_file, mode="w") as zf:
        zf.write(config_file, arcname="config.json")
        zf.write(csv_file, arcname="prompts.csv")
    print("Files inside ZIP:", zipfile.ZipFile(zip_file).namelist())

# 4. Parsing TOML configs (Used in pyproject.toml)
toml_text = '[training]\nepochs = 15\ndevice = "cuda"'
print("Parsed TOML:", tomllib.loads(toml_text))

# 5. SQLite3: Local SQL Database for saving AI Agent Memory
with sqlite3.connect(":memory:") as db:
    db.execute("CREATE TABLE chat_memory (id INTEGER PRIMARY KEY, role TEXT, message TEXT)")
    db.execute("INSERT INTO chat_memory (role, message) VALUES (?, ?)", ("user", "Explain AI"))
    saved_messages = db.execute("SELECT role, message FROM chat_memory").fetchall()
    print("Saved Chat Memory in SQLite:", saved_messages)

---
## Lesson 6: Standard Library Essentials for AI Pipelines[cite: 1, 2]

Python comes "batteries included" with powerful modules you don't even need to `pip install`:
* **`collections` (`Counter`, `defaultdict`, `deque`)**[cite: 2]:
  * `Counter`: Counts word/token frequencies in one line.
  * `deque(maxlen=N)`: A queue that automatically forgets the oldest item when full—perfect for keeping only the last $N$ messages in a chatbot's memory!
* **`re` (Regular Expressions)**[cite: 1, 2]: Pattern matching to strip out emails, URLs, or messy symbols from text before feeding it to an LLM.
* **`hashlib`**[cite: 2]: Creates unique fingerprints (`SHA-256`) of documents so your Vector Database never indexes the same PDF twice.
* **`random`, `math`, `statistics`**[cite: 2]: Setting random seeds (`random.seed(42)`) for reproducible experiments and calculating quick metrics.
* **`argparse` & `logging`**[cite: 2]: Accepting terminal flags (`--lr 0.001`) and recording timestamped logs instead of messy `print()` statements.

In [ ]:
import argparse
import hashlib
import logging
import math
import random
import re
import statistics
from collections import Counter, defaultdict, deque
from datetime import datetime
from zoneinfo import ZoneInfo

# 1. collections.Counter (Word frequency) & deque (Sliding window chatbot memory)
tokens = ["python", "ai", "python", "llm", "ai", "python"]
word_counts = Counter(tokens)
print("Most common tokens:", word_counts.most_common(2))

recent_memory = deque(maxlen=2)  # Only remembers the last 2 messages!
recent_memory.append("Message 1: Hi")
recent_memory.append("Message 2: How are you?")
recent_memory.append("Message 3: Teach me AI")  # Message 1 is automatically dropped!
print("Chatbot Sliding Window Memory:", list(recent_memory))

# 2. Regular Expressions (re) for cleaning messy text + hashlib for document ID
messy_text = "Order #4921: Call +92-300-1234567 or visit https://example.com"
clean_text = re.sub(r"https?://\S+", "[LINK]", messy_text)  # Replaces URLs with [LINK]
doc_id = hashlib.sha256(clean_text.encode()).hexdigest()[:12]
print("Cleaned Text:", clean_text, "| Unique Doc ID:", doc_id)

# 3. Reproducible Randomness & Basic Statistics
random.seed(42)  # Guarantees we get the exact same random numbers every run!
sample_scores = [round(random.uniform(0.70, 0.99), 3) for _ in range(5)]
print("Sampled Scores:", sample_scores)
print("Mean Score:", round(statistics.mean(sample_scores), 3), "| Sqrt(16):", math.sqrt(16))

# 4. Timezone-aware timestamps (datetime + zoneinfo)
pkt_time = datetime.now(ZoneInfo("Asia/Karachi")).strftime("%Y-%m-%d %H:%M:%S %Z")
print("Experiment Timestamp:", pkt_time)

# 5. Professional Logging & CLI Argument Parsing (argparse)
logging.basicConfig(level=logging.INFO, format="[%(levelname)s] %(message)s")
parser = argparse.ArgumentParser()
parser.add_argument("--model", type=str, default="deepseek-r1")
parser.add_argument("--batch_size", type=int, default=32)
cli_args = parser.parse_args(["--model", "qwen-2.5", "--batch_size", "64"])
logging.info(f"Starting training with model={cli_args.model}, batch_size={cli_args.batch_size}")